# Задание 2. Анализ влияния скидок на объём продаж

По истории продаж оценим минимальную из наблюдавшихся скидок, при которой среднедневные продажи товара увеличивались не менее чем в два раза.


## Подход

Сравниваем среднее количество проданных единиц за день при разных скидках с периодами без скидок. 
Длительность периодов может отличаться, поэтому суммарные продажи для такого сравнения я бы брать не стала.

In [1]:
import pandas as pd

file = 'Тестовое_задание_Data_аналитик_ЦО_2025_2(1).xlsx'
sales = pd.read_excel(file, sheet_name='Данные для задачи 2')

print('Строк:', sales.shape[0])
sales.head()

Строк: 1220


,Дата,Код товара,Количество продано,Розничная цена,Цена с учётом скидки
0,2024-06-01,3226300004,9,271,271.0
1,2024-06-02,3226300004,15,271,271.0
2,2024-06-03,3226300004,12,271,216.8
3,2024-06-04,3226300004,17,271,216.8
4,2024-06-05,3226300004,13,271,216.8


## 1. Размер скидки

Рассчитываем разницу между розничной ценой и фактической ценой продажи в процентах. Округляем.


In [2]:
#Округляем и считаем процент скидки 

sales['Скидка %'] = sales.apply(
    lambda row: round(
        100 * (1 - row['Цена с учётом скидки'] / row['Розничная цена']),
        1
    ),
    axis=1
)

sales[['Дата', 'Код товара', 'Количество продано', 'Скидка %']].head(15)

,Дата,Код товара,Количество продано,Скидка %
0,2024-06-01,3226300004,9,0.0
1,2024-06-02,3226300004,15,0.0
2,2024-06-03,3226300004,12,20.0
3,2024-06-04,3226300004,17,20.0
4,2024-06-05,3226300004,13,20.0
5,2024-06-06,3226300004,18,20.0
6,2024-06-07,3226300004,7,0.0
7,2024-06-08,3226300004,9,0.0
8,2024-06-09,3226300004,12,0.0
9,2024-06-10,3226300004,6,0.0


## 2. Продажи по уровням скидки

Для каждого товара считаем среднее число проданных единиц в день и количество дней наблюдений.

In [3]:
#Группируем код товара и %скидки 
discount_stats = sales.groupby(['Код товара', 'Скидка %']).agg(Средние_продажи_в_день=('Количество продано', 'mean'),
    Дней=('Дата', 'count')).reset_index()

discount_stats.sort_values(['Код товара', 'Скидка %']).head(16)

,Код товара,Скидка %,Средние_продажи_в_день,Дней
0,3226300004,0.0,10.224490,98
1,3226300004,20.0,15.000000,4
2,3226300004,25.0,23.375000,8
3,3226300004,30.0,24.125000,8
4,3226300004,40.0,37.000000,4
5,3234800007,0.0,10.295918,98
6,3234800007,10.0,16.250000,4
7,3234800007,15.0,15.500000,4
8,3234800007,20.0,16.000000,4
9,3234800007,25.0,24.250000,4


## 3. Сравнение с продажами без скидки

Добавляем базовый уровень продаж отдельно для каждого товара и рассчитываем отношение к нему.

In [8]:
#добавляем значение продаж без скидки
base = discount_stats[discount_stats['Скидка %'] == 0][['Код товара', 'Средние_продажи_в_день']
    ].rename(columns={'Средние_продажи_в_день': 'Продажи_без_скидки'})

comparison = pd.merge(discount_stats, base, on='Код товара', how='left')
comparison['Рост_к_базе'] = (comparison['Средние_продажи_в_день'] / comparison['Продажи_без_скидки'])

comparison.sort_values(['Код товара', 'Скидка %']).head(18)

,Код товара,Скидка %,Средние_продажи_в_день,Дней,Продажи_без_скидки,Рост_к_базе
0,3226300004,0.0,10.224490,98,10.224490,1.000000
1,3226300004,20.0,15.000000,4,10.224490,1.467066
2,3226300004,25.0,23.375000,8,10.224490,2.286178
3,3226300004,30.0,24.125000,8,10.224490,2.359531
4,3226300004,40.0,37.000000,4,10.224490,3.618762
5,3234800007,0.0,10.295918,98,10.295918,1.000000
6,3234800007,10.0,16.250000,4,10.295918,1.578295
7,3234800007,15.0,15.500000,4,10.295918,1.505451
8,3234800007,20.0,16.000000,4,10.295918,1.554014
9,3234800007,25.0,24.250000,4,10.295918,2.355302


## 4. Минимальная скидка

Оставляем уровни скидок, при которых среднедневные продажи выросли в 2x. 
Для каждого товара выбираем наименьшую из таких скидок.

In [5]:
# Забираем только те, где рост продаж  больше или равен 2 
twice = comparison[(comparison['Скидка %'] > 0) & (comparison['Рост_к_базе'] >= 2)]

recommendations = twice.groupby('Код товара').agg(Минимальная_скидка=('Скидка %', 'min')).reset_index()

# Добавляем фактические продажи и длительность наблюдений на выбранной скидке.
recommendations = pd.merge(
    recommendations,
    comparison[['Код товара', 'Скидка %', 'Средние_продажи_в_день',
                'Продажи_без_скидки', 'Рост_к_базе', 'Дней']],
    left_on=['Код товара', 'Минимальная_скидка'],
    right_on=['Код товара', 'Скидка %'],
    how='left'
)
recommendations.drop(columns=['Скидка %'], inplace=True)
recommendations.sort_values('Код товара')

,Код товара,Минимальная_скидка,Средние_продажи_в_день,Продажи_без_скидки,Рост_к_базе,Дней
0,3226300004,25.0,23.375,10.224490,2.286178,8
1,3234800007,25.0,24.250,10.295918,2.355302,4
2,14011400047,25.0,22.000,10.051020,2.188832,4
3,14012900005,25.0,26.500,9.561224,2.771612,4
4,15110900016,25.0,23.375,10.214286,2.288462,8
5,15111300006,25.0,25.750,9.441176,2.727414,4
6,15111300017,25.0,16.375,5.020408,3.261687,8
7,19000000363,25.0,26.250,10.479592,2.504869,4
8,19000013047,25.0,25.000,10.173469,2.457372,8
9,19760336592,25.0,25.875,10.142857,2.551056,8


## Вывод

По истории продаж для всех 10 товаров минимальная наблюдавшаяся скидка, при которой среднедневные продажи превышали уровень без скидки минимум вдвое, составляет **25%**.

Результат описывает историческую связь, а не гарантированный эффект от введения скидки. Для подтверждения стоит провести ограниченный тест; для оценки выгоды потребуются данные о себестоимости и маржинальности.